# Phase 4c: few-shot robustness + hybrid (fine-tuned encoder + few-shot LLM label)

**Settings:** Accelerator **GPU T4 x2** and **Internet on**.
**Attach (Add Input → Your Work):** `phase3-ablation`, `phase3a-llm` and `phase4b-eval` (all three).

1. Few-shot robustness: 2-shot with 3 more example sets, and 4-shot with 3 example sets (test set only).
2. The 2-shot LLM labels every train/val/test pair (the test part is reused from `phase4b-eval`).
3. Hybrid ablation for RoBERTa: base / +LLM (both reused) / +label / +LLM+label, 5 seeds.
4. Significance report with everything: `/kaggle/working/outputs/significance_weighted_f1.md`.

If the session stops, attach this notebook's last version too and run it again; finished work is skipped.

In [ ]:
!nvidia-smi -L
import transformers, torch
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU T4 x2, then run again. (On CPU every run takes hours.)"


In [ ]:
# Clone (or update) the project code and the dataset.
import os, subprocess, glob
REPO = "akashnegetive/stance-aware-llm-explanation-augmented-hate-speech-detection-for-replies-in-conversation"
CODE_DIR = "/kaggle/working/code"
if os.path.isdir(CODE_DIR):
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", CODE_DIR], check=True)
%cd {CODE_DIR}
if not os.path.isdir("data/counter_context"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/xinchenyu/counter_context", "data/counter_context"], check=True)

In [ ]:
import shlex
# Kaggle mounts notebook outputs at varying depths, so search all levels.
prev = sorted(glob.glob("/kaggle/input/**/outputs", recursive=True))
llm_files = sorted(glob.glob("/kaggle/input/**/llm/llm_outputs.jsonl", recursive=True))
prev_raw = sorted(glob.glob("/kaggle/input/**/llm_Qwen2.5-7B-Instruct_2shot/raw.jsonl", recursive=True))
assert prev and llm_files, "Attach phase3-ablation, phase3a-llm and phase4b-eval first"
LLM_FILE = llm_files[0]
LABEL_FILE = "/kaggle/working/outputs/llm_Qwen2.5-7B-Instruct_2shot/raw.jsonl"
DIRS = " ".join(shlex.quote(d) for d in prev + ["/kaggle/working/outputs"])
RESUME = " ".join(shlex.quote(p) for p in prev_raw)
print("Earlier results:", prev)
print("LLM file:", LLM_FILE)
print("2-shot test labels to reuse:", prev_raw or "none (test will be relabelled)")

In [ ]:
# Smoke test (~5-10 min, mostly the model download): 10 pairs with a new example set.
!python llm_classify.py --shots 2 --shot_seeds 1 --limit 10 --output_dir /kaggle/working/smoke
!head -n 3 /kaggle/working/smoke/llm_Qwen2.5-7B-Instruct_2shot_seed1_lim10/raw.jsonl

In [ ]:
# 1. Few-shot robustness on the test set (the original 2-shot draw, seed 0, comes from phase4b-eval).
!python llm_classify.py --shots 2 --shot_seeds 1 2 3 --output_dir /kaggle/working/outputs
!python llm_classify.py --shots 4 --shot_seeds 0 1 2 --output_dir /kaggle/working/outputs

In [ ]:
# 2. 2-shot LLM labels for every train/val/test pair (input for the hybrid).
!python llm_classify.py --shots 2 --shot_seeds 0 --splits train val test --output_dir /kaggle/working/outputs --resume_from {RESUME}

In [ ]:
# 3. Hybrid ablation, RoBERTa (lr 1e-5 from Phase 1b). base and llm are reused from phase3-ablation.
# To add DeBERTa (~2 h more), add microsoft/deberta-v3-base=2e-5 after roberta-base=1e-5.
!CUDA_VISIBLE_DEVICES=0 python ablation.py --models roberta-base=1e-5 --variants base llm label llm_label --select_metric weighted_f1 --llm_file {LLM_FILE} --llm_label_file {LABEL_FILE} --output_dir /kaggle/working/outputs --prev_results {DIRS}

In [ ]:
# 4. Significance tests + error analysis over everything (CPU, ~2 min).
!python significance.py --results_dirs {DIRS} --llm_file "{LLM_FILE}" --out_dir /kaggle/working/outputs